# Transformer Encoder

Reimplementation of the Transformer encoder block from [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929) (Dosovitskiy et al., 2020), using only basic PyTorch layers.

In this notebook I combine the multi-head self-attention layer from the previous notebook with an MLP into a single encoder block. Layer normalization is applied before every sub-block and a residual connection after every sub-block:

$$\mathbf{z}'_\ell = \text{MSA}(\text{LN}(\mathbf{z}_{\ell-1})) + \mathbf{z}_{\ell-1}, \qquad \ell = 1 \dots L \qquad \text{(Eq. 2)}$$

$$\mathbf{z}_\ell = \text{MLP}(\text{LN}(\mathbf{z}'_\ell)) + \mathbf{z}'_\ell, \qquad \ell = 1 \dots L \qquad \text{(Eq. 3)}$$

**Contents**
1. Setup
2. Data
3. Multi-head attention block
4. MLP block
5. Transformer encoder block

## 1. Setup

In [1]:
import torch
import torch.nn as nn

from torchinfo import summary

In [2]:
def set_seed(seed: int = 42):
    """Seeds PyTorch's CPU and CUDA random number generators for reproducible runs.

    Args:
        seed: Seed value passed to the generators.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

## 2. Data

The input is $\mathbf{z}_0$ from Eq. 1, the class token and embedded patches of a single image saved in the patch embedding notebook, with shape $(B, N+1, D) = (1, 257, 768)$.

In [3]:
x = torch.load("patch_embedding_sample.pt")

print(f"Shape of the patch embedding sample tensor: {x.shape}")
print(f"Patch embedding sample tensor: {x[:, :10]}")

Shape of the patch embedding sample tensor: torch.Size([1, 257, 768])
Patch embedding sample tensor: tensor([[[ 0.0109, -0.0094, -0.0243,  ..., -0.0091,  0.0090,  0.0024],
         [-0.4790,  0.1003, -0.1863,  ...,  0.4345, -0.2351,  0.2104],
         [-0.4672,  0.1485, -0.1737,  ...,  0.4147, -0.3149,  0.1660],
         ...,
         [-0.4990,  0.1077, -0.1480,  ...,  0.4059, -0.2472,  0.3080],
         [-0.3080,  0.1658, -0.1072,  ...,  0.2522, -0.1388,  0.1091],
         [-0.3206, -0.0556, -0.1652,  ...,  0.4020, -0.1186,  0.0959]]],
       grad_fn=<SliceBackward0>)


## 3. Multi-head attention block

The first sub-block normalizes the tokens and passes them through multi-head self-attention, the $\text{MSA}(\text{LN}(\mathbf{z}_{\ell-1}))$ term of Eq. 2. Dropout is applied after the output projection $W_O$, since Appendix B.1 applies it after every dense layer except the qkv projections. The residual connection is left to the encoder block in section 5.

### 3.1 Modules from the multi-head attention notebook

`ScaleDotProductAttention` and `MultiHeadAttention` are copied unchanged from the previous notebook.

In [4]:
class ScaleDotProductAttention(nn.Module):
    """Single attention head computing softmax(QK^T / sqrt(d_k))V over a sequence of tokens.

    Args:
        embedding_dim: Dimension D of the input tokens.
        d_k: Dimension of the queries and keys.
        d_v: Dimension of the values.
    """

    def __init__(self, embedding_dim: int, d_k: int, d_v: int):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.W_q = nn.Linear(embedding_dim, d_k)
        self.W_k = nn.Linear(embedding_dim, d_k)
        self.W_v = nn.Linear(embedding_dim, d_v)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Computes single-head self-attention.

        Args:
            x: Tensor of shape (B, N, embedding_dim).

        Returns:
            Tensor of shape (B, N, d_v).
        """
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        return output

In [5]:
class MultiHeadAttention(nn.Module):
    """Runs h attention heads in parallel, concatenates their outputs and projects them back to embedding_dim.

    Args:
        embedding_dim: Dimension D of the input tokens.
        d_k: Dimension of the queries and keys of each head.
        d_v: Dimension of the values of each head.
        h: Number of attention heads.
    """

    def __init__(self, embedding_dim: int, d_k: int, d_v: int, h: int):
        super().__init__()
        self.h = h
        self.d_k = d_k
        self.d_v = d_v
        self.attention_heads = nn.ModuleList([ScaleDotProductAttention(embedding_dim, d_k, d_v) for _ in range(h)])
        self.W_o = nn.Linear(h * d_v, embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Computes multi-head self-attention.

        Args:
            x: Tensor of shape (B, N, embedding_dim).

        Returns:
            Tensor of shape (B, N, embedding_dim).
        """
        head_outputs = [head(x) for head in self.attention_heads]
        multiple_head_output = torch.cat(head_outputs, dim=-1)
        output = self.W_o(multiple_head_output)
        return output

### 3.2 `MultiHeadAttentionBlock` module

In [6]:
class MultiHeadAttentionBlock(nn.Module):
    """Layer normalization followed by multi-head attention, the MSA(LN(z)) term of Eq. 2.

    Args:
        embedding_dim: Dimension D of the input tokens.
        d_k: Dimension of the queries and keys of each head.
        d_v: Dimension of the values of each head.
        h: Number of attention heads.
        dropout_rate: Dropout probability applied after the output projection W_o.
    """

    def __init__(self, embedding_dim: int, d_k: int, d_v: int, h: int, dropout_rate: float = 0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(embedding_dim, eps=1e-6)
        self.multi_head_attention = MultiHeadAttention(embedding_dim, d_k, d_v, h)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Applies layer normalization, multi-head self-attention, then dropout.

        The residual connection is added by the encoder block, not here.

        Args:
            x: Tensor of shape (B, N, embedding_dim).

        Returns:
            Tensor of shape (B, N, embedding_dim).
        """
        normalized = self.layer_norm(x)
        attention_output = self.multi_head_attention(normalized)
        output = self.dropout(attention_output)
        return output

In [7]:
set_seed()

multi_attention_block = MultiHeadAttentionBlock(embedding_dim=768, d_k=64, d_v=64, h=12, dropout_rate=0.1)
output = multi_attention_block(x)

print(f"Shape of the multi-head attention output tensor: {output.shape}")
print(f"Multi-head attention output tensor: {output[:, :10]}")

Shape of the multi-head attention output tensor: torch.Size([1, 257, 768])
Multi-head attention output tensor: tensor([[[ 0.3251, -0.1092,  0.1493,  ...,  0.2023,  0.2899, -0.3747],
         [ 0.0000, -0.0000,  0.1495,  ...,  0.2002,  0.2869, -0.0000],
         [ 0.3190, -0.1095,  0.1496,  ...,  0.2003,  0.0000, -0.3757],
         ...,
         [ 0.3193, -0.1095,  0.1504,  ...,  0.1999,  0.2872, -0.0000],
         [ 0.3182, -0.1094,  0.1501,  ...,  0.2001,  0.2874, -0.3757],
         [ 0.3191, -0.1094,  0.1499,  ...,  0.1999,  0.2865, -0.3764]]],
       grad_fn=<SliceBackward0>)


## 4. MLP block

The second sub-block is the $\text{MLP}(\text{LN}(\mathbf{z}'_\ell))$ term of Eq. 3. The MLP contains two linear layers with a GELU non-linearity, $\text{GELU}(x) = x\,\Phi(x)$, where $\Phi$ is the standard normal CDF:

$$\text{MLP}(\mathbf{x}) = \text{GELU}(\mathbf{x}W_1 + b_1)\,W_2 + b_2$$

The hidden layer expands each token from $D$ to the MLP size (3072 for ViT-Base, Table 1) and projects it back to $D$. Dropout is applied after every dense layer (Appendix B.1).

### 4.1 `MultiLayerPerceptronBlock` module

In [8]:
class MultiLayerPerceptronBlock(nn.Module):
    """Layer normalization followed by a multi-layer perceptron, the MLP(LN(z)) term of Eq. 3.

    Args:
        embedding_dim: Dimension D of the input tokens.
        mlp_dim: Dimension of the hidden layer in the MLP.
        dropout_rate: Dropout probability applied after each linear layer.
    """

    def __init__(self, embedding_dim: int, mlp_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(embedding_dim, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_dim, embedding_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Applies layer normalization, then a multi-layer perceptron.

        The residual connection is added by the encoder block, not here.

        Args:
            x: Tensor of shape (B, N, embedding_dim).

        Returns:
            Tensor of shape (B, N, embedding_dim).
        """
        normalized = self.layer_norm(x)
        output = self.mlp(normalized)
        return output

In [9]:
set_seed()

mlp_block = MultiLayerPerceptronBlock(embedding_dim=768, mlp_dim=3072, dropout_rate=0.1)
output = mlp_block(output+x)  # Adding the residual connection from the multi-head attention block

print(f"Shape of the MLP output tensor: {output.shape}")
print(f"MLP output tensor: {output[:, :10]}")

Shape of the MLP output tensor: torch.Size([1, 257, 768])
MLP output tensor: tensor([[[-0.0629,  0.1421,  0.1382,  ...,  0.3725,  0.0115, -0.4547],
         [ 0.0170, -0.1863,  0.1990,  ...,  0.2503, -0.2466, -0.2091],
         [ 0.0203, -0.1582,  0.1650,  ...,  0.4177, -0.0000, -0.2628],
         ...,
         [-0.1779, -0.2030, -0.0876,  ...,  0.1386,  0.0763, -0.4267],
         [ 0.0440, -0.2167,  0.1699,  ...,  0.0274, -0.1022, -0.5692],
         [-0.0011, -0.0565,  0.1466,  ...,  0.1112, -0.1047, -0.4028]]],
       grad_fn=<SliceBackward0>)


## 5. Transformer encoder block

The encoder block chains both sub-blocks and adds the residual connections of Eq. 2 and Eq. 3. Both sub-blocks map $(B, N+1, D)$ to $(B, N+1, D)$, so the block can be stacked $L$ times (12 for ViT-Base).

### 5.1 `TransformerEncoderBlock` module

In [10]:
class TransformerEncoderBlock(nn.Module):
    """A single block of the Transformer encoder, consisting of multi-head attention and a multi-layer perceptron.

    Args:
        embedding_dim: Dimension D of the input tokens.
        h: Number of attention heads.
        mlp_dim: Dimension of the hidden layer in the MLP.
        dropout_rate: Dropout probability applied after the attention output projection and each linear layer of the MLP.
    """

    def __init__(self, embedding_dim: int, h: int, mlp_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        assert embedding_dim % h == 0, f"embedding_dim must be divisible by h: embedding_dim={embedding_dim}, h={h}"
        d_k = embedding_dim // h
        d_v = embedding_dim // h
        self.mha_block = MultiHeadAttentionBlock(embedding_dim, d_k, d_v, h, dropout_rate)
        self.mlp_block = MultiLayerPerceptronBlock(embedding_dim, mlp_dim, dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Applies multi-head attention followed by a multi-layer perceptron, each with a residual connection.

        Args:
            x: Tensor of shape (B, N, embedding_dim).

        Returns:
            Tensor of shape (B, N, embedding_dim).
        """
        # Multi-head attention with residual connection
        mha_output = self.mha_block(x)
        x = x + mha_output

        # MLP with residual connection
        mlp_output = self.mlp_block(x)
        output = x + mlp_output

        return output

### 5.2 Model summary

In [11]:
set_seed()

transformer_encoder_block = TransformerEncoderBlock(embedding_dim=768, h=12, mlp_dim=3072, dropout_rate=0.1)

summary(
    model=transformer_encoder_block,
    input_size=(1, 257, 768),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
    
)

Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
TransformerEncoderBlock (TransformerEncoderBlock)       [1, 257, 768]        [1, 257, 768]        --                   True
├─MultiHeadAttentionBlock (mha_block)                   [1, 257, 768]        [1, 257, 768]        --                   True
│    └─LayerNorm (layer_norm)                           [1, 257, 768]        [1, 257, 768]        1,536                True
│    └─MultiHeadAttention (multi_head_attention)        [1, 257, 768]        [1, 257, 768]        --                   True
│    │    └─ModuleList (attention_heads)                --                   --                   1,771,776            True
│    │    └─Linear (W_o)                                [1, 257, 768]        [1, 257, 768]        590,592              True
│    └─Dropout (dropout)                                [1, 257, 768]        [1, 257, 768]        --                   --
├─Mul